
## CONCEPT-FIRST SUMMARIZATION

This notebook demonstrates a concept-first summarization approach, where the underlying idea is extracted before generating the summary.

Instead of summarizing the original text directly, we first extract the underlying concept and then generate the final output from that concept.

Traditional summarization models often compress text by reusing phrases and sentence structures from the source material. This can lead to outputs that feel like shortened copies rather than truly novel summaries.

This approach addresses that by:
1. **Concept Extraction**: First, we extract the core idea from the text
2. **Summary Generation**: Then, we generate a summary from that concept rather than from the original text

This produces more original, paraphrased summaries that don't rely on copying from the source.

## USE THIS NOTEBOOK IF:
- You want summaries that sound more natural and less like compressed copies
- You need multiple summary styles (casual, formal, bullet points, paragraphs)
- You want to minimize copying from the source material


## Summary Modes

| Mode      | What it should do                                 |
| --------- | ------------------------------------------------- |
| Casual    | explain simply                                    |
| Formal    | expand + elevate language                         |
| Bullet    | sentence decomposition → paraphrase → compression |
| Paragraph | sentence fusion → restructuring                   |


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.utils import logging as hf_logging
import re

# =====================================================
# MODELS
# =====================================================

BART_MODEL = "facebook/bart-large-cnn"
T5_MODEL = "google/flan-t5-base"
BULLET = "\u2022"

# Hide noisy Transformers warnings without changing model configs or weights.
hf_logging.set_verbosity_error()

print("Loading BART...")
bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL)

print("Loading FLAN-T5...")
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

# =====================================================
# TEXT UTILITIES
# =====================================================


def word_count(text):
    return len(text.split())


def clean_word_count(text):
    return len(get_words(text))


def get_paragraphs(text):
    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n", text)
        if paragraph.strip()
    ]

    if not paragraphs:
        return [text.strip()]

    return paragraphs


def split_into_sentences(text):
    normalized = re.sub(r"\s+", " ", text.strip())

    if not normalized:
        return []

    sentences = re.findall(r"[^.!?]+[.!?]+(?=\s|$)", normalized)
    trailing = re.sub(r".*[.!?]+\s*", "", normalized)

    if trailing and trailing != normalized:
        sentences.append(trailing)

    if not sentences:
        return [normalized]

    return [sentence.strip() for sentence in sentences if sentence.strip()]


def split_into_chunks(text, chunk_size=400):
    words = text.split()

    return [
        " ".join(words[index:index + chunk_size])
        for index in range(0, len(words), chunk_size)
    ]


def ensure_sentence(text):
    text = " ".join(text.strip().split())

    if not text:
        return text

    if text[-1] not in ".!?":
        text += "."

    return text


def lowercase_first(text):
    if not text:
        return text

    return text[0].lower() + text[1:]


def strip_list_marker(text):
    marker_chars = "-*" + BULLET + "0123456789. )\t"
    return text.lstrip(marker_chars).strip()


def force_one_sentence(text):
    text = strip_list_marker(" ".join(text.split()))
    sentences = split_into_sentences(text)

    if len(sentences) <= 1:
        return ensure_sentence(text)

    fused = "; ".join(
        sentence.rstrip(".!?")
        for sentence in sentences
    )

    return ensure_sentence(fused)


def strip_summary_intro(text):
    text = force_one_sentence(text).rstrip(".!?")
    intro_patterns = [
        r"^this\s+(passage|paragraph|text|article)\s+"
        r"(talks about(?: how)?|gives you an idea about|gives an idea about|"
        r"discusses(?: how)?|explains|describes|covers|is about)\s+",
        r"^the\s+(passage|paragraph|text|article)\s+"
        r"(talks about(?: how)?|gives you an idea about|gives an idea about|"
        r"discusses(?: how)?|explains|describes|covers|is about)\s+",
    ]

    for pattern in intro_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    return text


def apply_summary_prefix(text, prefix):
    body = strip_summary_intro(text)

    if not body:
        return ensure_sentence(prefix)

    if prefix.lower().endswith(" how") and body.lower().startswith("how "):
        body = body[4:].strip()

    return ensure_sentence(f"{prefix} {lowercase_first(body)}")


def is_meta_summary(text):
    lowered = text.lower()
    meta_phrases = [
        "a short summary",
        "brief summary",
        "main idea of this passage",
        "main idea of the passage",
        "main idea of this text",
        "what the passage is about",
        "what this passage is about",
        "what the text is about",
        "the passage talks about",
        "the text talks about",
        "this passage is about",
        "this text is about",
    ]

    return any(phrase in lowered for phrase in meta_phrases)


def format_with_newlines(text, word_limit=20):
    words = text.split()
    formatted_lines = []
    current_line_words = []
    for i, word in enumerate(words):
        current_line_words.append(word)
        if (i + 1) % word_limit == 0:
            formatted_lines.append(" ".join(current_line_words))
            current_line_words = []
    if current_line_words:
        formatted_lines.append(" ".join(current_line_words))
    return "\n".join(formatted_lines)

# =====================================================
# COPY DETECTION
# =====================================================


def get_words(text):
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def get_ngrams(text, n=4):
    words = get_words(text)

    return {
        tuple(words[index:index + n])
        for index in range(len(words) - n + 1)
    }


def copied_ngram_ratio(source, candidate, n=4):
    candidate_ngrams = get_ngrams(candidate, n)

    if not candidate_ngrams:
        return 0

    source_ngrams = get_ngrams(source, n)
    copied_ngrams = candidate_ngrams & source_ngrams

    return len(copied_ngrams) / len(candidate_ngrams)


def choose_least_copied(source, candidates):
    usable = [
        force_one_sentence(candidate)
        for candidate in candidates
        if candidate and candidate.strip()
    ]

    if not usable:
        return ""

    return min(
        usable,
        key=lambda candidate: copied_ngram_ratio(source, candidate)
    )

# =====================================================
# FLAN-T5 GENERATION
# =====================================================


def t5_generate(
    prompt,
    max_tokens=64,
    num_return_sequences=1,
    temperature=0.9,
):
    input_ids = t5_tokenizer.encode(prompt, return_tensors="pt")
    output_ids = t5_model.generate(
        input_ids,
        max_length=max_tokens,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        do_sample=True,
    )
    return [t5_tokenizer.decode(output_id, skip_special_tokens=True) for output_id in output_ids]


# =====================================================
# CONCEPT-FIRST SUMMARIZATION
# =====================================================


def concept_first_summarize(text, mode="casual"):
    # Step 1: Extract the concept
    concept_prompt = f"Extract the main concept from this text in one sentence:\n{text}"
    concepts = t5_generate(concept_prompt, max_tokens=32, num_return_sequences=1)
    concept = concepts[0] if concepts else ""

    # Step 2: Generate summary from the concept
    if mode == "casual":
        summary_prompt = f"Explain this concept simply:\n{concept}"
    elif mode == "formal":
        summary_prompt = f"Explain this concept in formal language with elevated vocabulary:\n{concept}"
    elif mode == "bullet":
        summary_prompt = f"Break down this concept into 3-4 key bullet points:\n{concept}"
    elif mode == "paragraph":
        summary_prompt = f"Expand this concept into a full paragraph:\n{concept}"
    else:
        summary_prompt = f"Summarize this concept:\n{concept}"

    summaries = t5_generate(summary_prompt, max_tokens=128, num_return_sequences=3)

    # Step 3: Choose the least copied version
    best_summary = choose_least_copied(text, summaries)

    return {
        "concept": concept,
        "summary": best_summary,
        "mode": mode,
        "copy_ratio": copied_ngram_ratio(text, best_summary)
    }


# =====================================================
# EXAMPLE USAGE
# =====================================================

if __name__ == "__main__":
    sample_text = """
    Machine learning is a subset of artificial intelligence that focuses on the development
    of algorithms and statistical models that enable computers to learn and improve from experience
    without being explicitly programmed. The algorithms can identify patterns in data, make decisions,
    and improve their performance through iterative processes.
    """

    modes = ["casual", "formal", "bullet", "paragraph"]
    
    for mode in modes:
        result = concept_first_summarize(sample_text, mode=mode)
        print(f"\n{'='*60}")
        print(f"Mode: {mode.upper()}")
        print(f"{'='*60}")
        print(f"Concept: {result['concept']}")
        print(f"\nSummary: {result['summary']}")
        print(f"\nCopy Ratio: {result['copy_ratio']:.2%}")
